# Mondrian with predicted groups — deployment hit when the group is estimated, not given

Reuses cached head posteriors + the recoverability probe (no retraining). Per (backbone × dataset)
× all training methods × APS (+RAPS/THR appendix), ρ_cal=0.95. Predict â for every cal/test point,
then compare Mondrian three ways — **(a)** true groups, **(b)** predicted groups at TEST only,
**(c)** predicted at cal AND test (a never observed) — with coverage scored on TRUE groups.

Reports per method: worst-group coverage + set size under (a)/(b)/(c) + **gap (a)−(c)**. Expectation:
â is accurate → (c) ≈ (a) on worst-group coverage (small gap), modest efficiency cost → Mondrian is
deployable without ground-truth groups. Reported honestly if the gap is large. **STOP** after.

## 0. Parameters — **EDIT THESE**

In [ ]:
REPO_SOURCE   = "git"
REPO_URL      = "https://github.com/octadion/vgscp.git"
REPO_BRANCH   = "main"
REPO_DRIVE_ZIP= "/content/drive/MyDrive/vgscp.zip"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"
SEEDS         = 3
N_SPLITS      = 10
CELEBA_RESNET_MAX_TRAIN = 30000
WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE = "kaggle"   # "kaggle" (needs kaggle.json) | "drive" | "skip"
CELEBA_DRIVE  = ""
import os, sys, time, subprocess
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)

## 1. GPU + install

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn scipy pandas matplotlib torchvision", shell=True)

## 2. Mount Drive + repo + datasets

In [ ]:
from google.colab import drive
drive.mount("/content/drive"); os.makedirs(DRIVE_CACHE, exist_ok=True)
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba
os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK: os.environ["CELEBA_ROOT"] = CELEBA_ROOT; print("CelebA root:", CELEBA_ROOT)
else: print(f"[note] CelebA unavailable (source={CELEBA_SOURCE}) -> Waterbirds-only.")
for c in ("cache_clip", "cache_resnet", "study"):
    sh(f"rm -rf results/{c}"); os.makedirs(f"{DRIVE_CACHE}/{c}", exist_ok=True); os.makedirs("results", exist_ok=True)
    sh(f"ln -s {DRIVE_CACHE}/{c} results/{c}")

## 3. Build GridData per (backbone × dataset) — cache hit if features already extracted

In [ ]:
from study_robust_train.datasets import build_griddata

def cfg_for(dataset):
    base = {"clip": {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cuda",
                     "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cuda", "epochs": 10, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224, "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
        base["resnet"]["max_train"] = CELEBA_RESNET_MAX_TRAIN
    return base

KEYS = [("waterbirds", "resnet50_erm"), ("waterbirds", "clip_vitb32")]
if CELEBA_OK: KEYS += [("celeba", "resnet50_erm"), ("celeba", "clip_vitb32")]
data, skipped = {}, []
for ds, bb in KEYS:
    try:
        t = time.time(); data[(bb, ds)] = build_griddata(ds, bb, cfg_for(ds), seed=0)
        print(f"[built] {bb}/{ds} ({(time.time()-t)/60:.1f} min)")
    except Exception as e:
        skipped.append((bb, ds)); print(f"[SKIP] {bb}/{ds}: {e}")
print("cells:", list(data.keys()), "| skipped:", skipped)

## 4. Run predicted-group Mondrian (a / b / c) + gap → write MD + CSV → STOP

In [ ]:
from study_robust_train.predicted_group_mondrian import run_predicted_group, write_csv, write_md

t = time.time()
out = run_predicted_group(data, seeds=tuple(range(SEEDS)), n_splits=N_SPLITS)  # APS/RAPS/THR, ρ_cal=ρ_test=0.95
print(f"[predicted-group] {len(out['records'])} records, {len(out['excluded'])} excluded ({(time.time()-t)/60:.1f} min)")
os.makedirs("results/study", exist_ok=True)
write_csv(out["records"], "results/study/predicted_group_mondrian.csv")
write_md(out, "PREDICTED_GROUP_MONDRIAN.md")
print("wrote PREDICTED_GROUP_MONDRIAN.md, results/study/predicted_group_mondrian.csv")

for key, v in out["verdicts"].items():
    print(f"\n{key}: all methods deployable (|cov(a)-cov(c)|<=0.02): {v['all_deployable']}")
    for m, row in v["methods"].items():
        print(f"  {m}: AUROC={row['probe_auroc']:.3f}  cov(a)={row['a_true']['cov']['mean']:.3f}  "
              f"cov(b)={row['b_pred_test']['cov']['mean']:.3f}  cov(c)={row['c_pred_both']['cov']['mean']:.3f}  "
              f"gap(a-c)={row['gap_cov_a_minus_c']:+.3f}  size cost={row['gap_size_c_minus_a']:+.2f}")

## 5. Show PREDICTED_GROUP_MONDRIAN.md

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open("PREDICTED_GROUP_MONDRIAN.md", encoding="utf-8").read()))

## 6. STOP — predicted-group Mondrian complete
If gap(a−c) is small for all methods, group-conditional calibration is deployable without
ground-truth group labels (â suffices). Hand the table to the researcher. **STOP for human review** —
do NOT run the full-GroupDRO fine-tune or any 3rd/4th dataset.